# ABSA Sentiment Analysis — Bangkok BTS Skytrain Reviews

**Phase 2 of CRISP-DM**: aspect-based sentiment classification on cleaned BTS reviews.

**Pipeline**:
1. Load `all_reviews_cleaned.csv` (10 canonical aspects, 3-class sentiment)
2. Group-aware split (no leakage by `text_id`)
3. Train **TF-IDF + LogReg** (Traditional ML) and **DistilBERT** (Deep Learning)
4. Evaluate w/ Confusion Matrix + Precision/Recall + per-aspect breakdown
5. Visualize: NSS trend, word clouds, LDA topics
6. Inference helper: rule-based aspect → ABSA sentiment

All artifacts saved to `/kaggle/working/artifacts/`.

## 1. Setup

In [ ]:
# Kaggle has most deps pre-installed; only wordcloud may need a pin
!pip install -q wordcloud imbalanced-learn

In [ ]:
from __future__ import annotations

import hashlib
import json
import logging
import random
import re
import time
from collections import Counter
from contextlib import ContextDecorator
from dataclasses import dataclass
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

## 2. Config — single source of truth

In [ ]:
# Adjust CSV_PATH if your Kaggle dataset slug differs
INPUT_CANDIDATES = [
    Path("/kaggle/input/bts-reviews/all_reviews_cleaned.csv"),
    Path("/kaggle/input/all-reviews-cleaned/all_reviews_cleaned.csv"),
    Path("all_reviews_cleaned.csv"),  # fallback for local runs
]
CSV_PATH = next((p for p in INPUT_CANDIDATES if p.exists()), INPUT_CANDIDATES[0])
print(f"CSV_PATH = {CSV_PATH}  (exists: {CSV_PATH.exists()})")

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
ARTIFACTS = ROOT / "artifacts"
MODELS_DIR = ARTIFACTS / "models"
FIGURES_DIR = ARTIFACTS / "figures"
LOGS_DIR = ARTIFACTS / "logs"
REPORTS_DIR = ARTIFACTS / "reports"
for d in (MODELS_DIR, FIGURES_DIR, LOGS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

BASELINE_PATH = MODELS_DIR / "baseline.joblib"
TRANSFORMER_DIR = MODELS_DIR / "distilbert"

SENTIMENT_LABELS = ["Negative", "Neutral", "Positive"]
LABEL2ID = {l: i for i, l in enumerate(SENTIMENT_LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

CANONICAL_ASPECTS = [
    "Fare & Payment System", "Crowding & Comfort", "Punctuality & Reliability",
    "Route Coverage & Connectivity", "Facilities & Accessibility",
    "Safety & Security", "Information & Navigation", "Staff & Service Quality",
    "Cleanliness & Hygiene", "Overall Experience & Convenience",
]

# Order matters — first match wins
ASPECT_RULES = [
    ("Fare & Payment System",
        r"fare|price|cost|ticket|payment|pay|cheap|expensive|rabbit card|top.?up|fee|baht|surcharge"),
    ("Crowding & Comfort",
        r"crowd|crowded|packed|rush hour|peak|seat|standing|space|comfort|comfortable|squeeze|full|busy"),
    ("Cleanliness & Hygiene",
        r"clean|dirty|hygiene|smell|odor|trash|garbage|litter|sanit"),
    ("Staff & Service Quality",
        r"staff|officer|guard|employee|service|rude|helpful|friendly|assist|attitude|personnel"),
    ("Punctuality & Reliability",
        r"delay|late|on time|punctual|reliable|schedule|frequency|wait|waiting|interval|breakdown|cancel"),
    ("Safety & Security",
        r"safe|safety|security|accident|crime|theft|steal|cctv|police|emergency|danger|hazard"),
    ("Facilities & Accessibility",
        r"accessib|disable|wheelchair|elevator|lift|escalator|ramp|elderly|handicap|barrier|"
        r"facilit|infrastructure|platform|toilet|restroom|wifi|air.?con|bench|repair|maintain|maintenance"),
    ("Information & Navigation",
        r"sign|signage|map|direction|navigate|navigation|confus|lost|label|display|board|announcement|"
        r"data|information|info|app|website|online|real.?time|update|timetable|screen"),
    ("Route Coverage & Connectivity",
        r"route|coverage|connect|extend|extension|line|network|interchange|transfer|reach|destination|suburb"),
    ("Overall Experience & Convenience",
        r"convenient|convenience|easy|simple|quick|fast|efficient|hassle|smooth|seamless|"
        r"overall|experience|general|impression|recommend|worth|value|enjoy|love|hate|terrible|excellent|"
        r"great|good|bad|poor|amazing|awful"),
]
ASPECT_FALLBACK = "Overall Experience & Convenience"

RANDOM_SEED = 42
SPLIT_N_FOLDS = 5

TFIDF_MAX_FEATURES = 80_000
TFIDF_NGRAM = (1, 2)
LOGREG_C = 2.0
LOGREG_MAX_ITER = 2000

TRANSFORMER_MODEL = "distilbert-base-uncased"
TRANSFORMER_EPOCHS = 3.0
TRANSFORMER_BATCH = 32  # bumped for Kaggle GPU
TRANSFORMER_LR = 2e-5
TRANSFORMER_MAX_LEN = 256
TRANSFORMER_PATIENCE = 2

LDA_N_TOPICS = 6
LDA_N_TOP_WORDS = 12
WORDCLOUD_MAX_WORDS = 150

## 3. Utils — logging, seed, JSON, Timer

In [ ]:
def setup_logging(log_path: Path, level: int = logging.INFO) -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("bts")
    logger.setLevel(level)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
    fh = logging.FileHandler(log_path, mode="a", encoding="utf-8")
    fh.setFormatter(fmt)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(sh)
    return logger


def set_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


class Timer(ContextDecorator):
    def __init__(self, label: str, logger: logging.Logger | None = None):
        self.label = label
        self.logger = logger or logging.getLogger("bts")
        self.elapsed = 0.0

    def __enter__(self):
        self.logger.info(f"[start] {self.label}")
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *_):
        self.elapsed = time.perf_counter() - self._t0
        self.logger.info(f"[done]  {self.label} ({self.elapsed:.2f}s)")


log = setup_logging(LOGS_DIR / "training.log")
set_seeds(RANDOM_SEED)
log.info("setup ok")

## 4. Data — load, group-aware split, sentence-pair format

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold


def _stable_text_id(text: str) -> str:
    # Same review across rows gets the same id — used to prevent split leakage
    normalized = " ".join((text or "").split()).strip().lower()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()[:16]


def load_data(csv_path: Path = CSV_PATH) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required = {"review_text", "review_rating", "aspect", "sentiment"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    df = df.copy()
    df["sentiment"] = df["sentiment"].astype(str).str.strip()
    df["aspect"] = df["aspect"].astype(str).str.strip()
    df = df[df["sentiment"].isin(SENTIMENT_LABELS)].reset_index(drop=True)
    df = df[df["aspect"].isin(CANONICAL_ASPECTS)].reset_index(drop=True)
    df["text_id"] = df["review_text"].astype(str).map(_stable_text_id)
    return df


@dataclass(frozen=True)
class Splits:
    train: pd.DataFrame
    val: pd.DataFrame
    test: pd.DataFrame


def make_splits(df: pd.DataFrame, *, n_splits: int = SPLIT_N_FOLDS, seed: int = RANDOM_SEED) -> Splits:
    # fold0 → test, fold1 → val, rest → train; grouped by text_id, stratified by sentiment
    y = df["sentiment"].to_numpy()
    groups = df["text_id"].to_numpy()
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = list(sgkf.split(np.zeros(len(df)), y, groups))

    test_ids = set(df.iloc[folds[0][1]]["text_id"].tolist())
    val_ids = set(df.iloc[folds[1][1]]["text_id"].tolist()) - test_ids
    is_test = df["text_id"].isin(test_ids)
    is_val = df["text_id"].isin(val_ids)

    train = df[~is_test & ~is_val].reset_index(drop=True)
    val = df[is_val].reset_index(drop=True)
    test = df[is_test].reset_index(drop=True)

    overlap = (set(train["text_id"]) & set(val["text_id"])) | \
              (set(train["text_id"]) & set(test["text_id"])) | \
              (set(val["text_id"]) & set(test["text_id"]))
    if overlap:
        raise RuntimeError(f"Group leakage: {len(overlap)} overlapping text_ids")
    return Splits(train=train, val=val, test=test)


def format_pair(aspect: str, text: str) -> str:
    return f"Aspect: {aspect}\nReview: {text}"


def build_xy(df: pd.DataFrame, *, aspect_col: str = "aspect"):
    x = df.apply(lambda r: format_pair(r[aspect_col], r["review_text"]), axis=1).to_numpy()
    y = df["sentiment"].to_numpy()
    return x, y


with Timer("load + split", log):
    df = load_data()
    splits = make_splits(df)
    log.info(f"rows: total={len(df)} train={len(splits.train)} val={len(splits.val)} test={len(splits.test)}")

x_train, y_train = build_xy(splits.train)
x_val, y_val = build_xy(splits.val)
x_test, y_test = build_xy(splits.test)

## 5. Aspect extractor — rule-based, used for inference

In [ ]:
_COMPILED_RULES = [(a, re.compile(p, re.IGNORECASE)) for a, p in ASPECT_RULES]


def extract_aspect(text: str) -> str:
    s = str(text)
    for aspect, pattern in _COMPILED_RULES:
        if pattern.search(s):
            return aspect
    return ASPECT_FALLBACK


# Quick sanity check
print(extract_aspect("The BTS was packed during rush hour"))
print(extract_aspect("Tickets are too expensive"))
print(extract_aspect("Random nonsense xyz"))

## 6. Baseline — TF-IDF + LogReg (Traditional ML)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def build_baseline() -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=TFIDF_MAX_FEATURES,
            ngram_range=TFIDF_NGRAM,
            sublinear_tf=True,
            min_df=2,
        )),
        ("logreg", LogisticRegression(
            C=LOGREG_C,
            max_iter=LOGREG_MAX_ITER,
            class_weight="balanced",
            random_state=RANDOM_SEED,
        )),
    ])


with Timer("train baseline", log):
    baseline = build_baseline()
    baseline.fit(x_train, y_train)
    joblib.dump(baseline, BASELINE_PATH)
    log.info(f"saved → {BASELINE_PATH}")

## 7. Transformer — DistilBERT fine-tune (Deep Learning)

In [ ]:
import torch
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


def _to_hf(df: pd.DataFrame) -> Dataset:
    return Dataset.from_dict({
        "aspect": df["aspect"].astype(str).tolist(),
        "text": df["review_text"].astype(str).tolist(),
        "labels": [LABEL2ID[s] for s in df["sentiment"].astype(str).tolist()],
    })


def train_transformer(train_df: pd.DataFrame, val_df: pd.DataFrame,
                       *, epochs: float = TRANSFORMER_EPOCHS) -> Path:
    out_dir = TRANSFORMER_DIR
    out_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL, use_fast=True)

    def tokenize(batch):
        return tokenizer(batch["aspect"], batch["text"],
                         truncation=True, max_length=TRANSFORMER_MAX_LEN)

    train_ds = _to_hf(train_df).map(tokenize, batched=True, remove_columns=["aspect", "text"])
    val_ds = _to_hf(val_df).map(tokenize, batched=True, remove_columns=["aspect", "text"])

    # Class-weighted CE compensates for the imbalanced 53/22/25 distribution
    y_tr = np.array([LABEL2ID[s] for s in train_df["sentiment"].astype(str).tolist()])
    cw = compute_class_weight(class_weight="balanced",
                              classes=np.arange(len(SENTIMENT_LABELS)), y=y_tr)
    class_weights = torch.tensor(cw, dtype=torch.float)

    model = AutoModelForSequenceClassification.from_pretrained(
        TRANSFORMER_MODEL,
        num_labels=len(SENTIMENT_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(labels, preds, average="weighted", zero_division=0)),
        }

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.get("labels")
            outputs = model(**inputs)
            logits = outputs.get("logits")
            loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    args = TrainingArguments(
        output_dir=str(out_dir),
        learning_rate=TRANSFORMER_LR,
        per_device_train_batch_size=TRANSFORMER_BATCH,
        per_device_eval_batch_size=TRANSFORMER_BATCH,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=2,
        report_to=[],
        seed=RANDOM_SEED,
        fp16=torch.cuda.is_available(),
    )

    # `processing_class` replaces `tokenizer` in transformers >= 5
    trainer = WeightedTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=TRANSFORMER_PATIENCE)],
    )
    trainer.train()
    trainer.save_model(str(out_dir))
    tokenizer.save_pretrained(str(out_dir))
    return out_dir


def predict_transformer(df: pd.DataFrame, *, model_dir: Path = TRANSFORMER_DIR,
                       batch_size: int = 64):
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    aspects = df["aspect"].astype(str).tolist()
    texts = df["review_text"].astype(str).tolist()
    id2label = {int(k): v for k, v in model.config.id2label.items()}

    labels, probs_all = [], []
    for i in range(0, len(df), batch_size):
        a = aspects[i : i + batch_size]
        t = texts[i : i + batch_size]
        enc = tokenizer(a, t, truncation=True, max_length=TRANSFORMER_MAX_LEN,
                        padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        labels.extend([id2label[j] for j in logits.argmax(dim=-1).cpu().numpy().tolist()])
        probs_all.append(probs)
    return np.array(labels), np.concatenate(probs_all, axis=0) if probs_all else np.zeros((0, len(id2label)))


with Timer("train transformer", log):
    train_transformer(splits.train, splits.val)

## 8. Evaluation — Confusion Matrix + Precision/Recall + per-aspect

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def evaluate(name: str, y_true, y_pred) -> dict:
    return {
        "name": name,
        "labels": SENTIMENT_LABELS,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=SENTIMENT_LABELS).tolist(),
        "report": classification_report(y_true, y_pred, labels=SENTIMENT_LABELS,
                                         output_dict=True, zero_division=0),
    }


def evaluate_by_aspect(df: pd.DataFrame, y_true, y_pred, *, min_rows: int = 30) -> dict:
    df = df.reset_index(drop=True)
    out: dict[str, dict] = {}
    for aspect, sub in df.groupby("aspect"):
        if len(sub) < min_rows:
            continue
        idx = sub.index.to_numpy()
        out[str(aspect)] = {
            "rows": int(len(sub)),
            "accuracy": float(accuracy_score(y_true[idx], y_pred[idx])),
            "macro_f1": float(f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0)),
        }
    return out


def error_analysis(df: pd.DataFrame, y_true, y_pred, n: int = 30) -> pd.DataFrame:
    df = df.reset_index(drop=True).copy()
    mask = y_true != y_pred
    errs = df.loc[mask].copy()
    errs["y_true"] = y_true[mask]
    errs["y_pred"] = y_pred[mask]
    errs["text_preview"] = errs["review_text"].astype(str).str.slice(0, 180)
    return errs[["aspect", "y_true", "y_pred", "text_preview"]].head(n)


def plot_confusion_matrix(y_true, y_pred, *, title: str, out_path: Path) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=SENTIMENT_LABELS)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=SENTIMENT_LABELS, yticklabels=SENTIMENT_LABELS, ax=axes[0])
    axes[0].set(title=f"{title} — counts", xlabel="Predicted", ylabel="True")
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=SENTIMENT_LABELS, yticklabels=SENTIMENT_LABELS, ax=axes[1], vmin=0, vmax=1)
    axes[1].set(title=f"{title} — normalized", xlabel="Predicted", ylabel="True")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    plt.close()


def plot_per_class_pr(report: dict, *, title: str, out_path: Path) -> None:
    metrics = ["precision", "recall", "f1-score"]
    data = np.array([[report[l][m] for m in metrics] for l in SENTIMENT_LABELS])
    x = np.arange(len(SENTIMENT_LABELS))
    width = 0.25
    plt.figure(figsize=(9, 5))
    for i, m in enumerate(metrics):
        plt.bar(x + (i - 1) * width, data[:, i], width, label=m)
    plt.xticks(x, SENTIMENT_LABELS)
    plt.ylim(0, 1)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    plt.close()


results: dict = {
    "data": {
        "rows_total": int(len(df)),
        "rows_train": int(len(splits.train)),
        "rows_val": int(len(splits.val)),
        "rows_test": int(len(splits.test)),
    }
}

# ── Baseline
with Timer("evaluate baseline", log):
    val_pred = baseline.predict(x_val)
    test_pred = baseline.predict(x_test)
    bl = {
        "val": {**evaluate("baseline_val", y_val, val_pred),
                "by_aspect": evaluate_by_aspect(splits.val, y_val, val_pred)},
        "test": {**evaluate("baseline_test", y_test, test_pred),
                 "by_aspect": evaluate_by_aspect(splits.test, y_test, test_pred)},
    }
    results["baseline"] = bl
    log.info(f"baseline test: acc={bl['test']['accuracy']:.4f} macro_f1={bl['test']['macro_f1']:.4f}")
    plot_confusion_matrix(y_test, test_pred, title="baseline_test",
                          out_path=FIGURES_DIR / "cm_baseline_test.png")
    plot_per_class_pr(bl["test"]["report"], title="baseline_test",
                      out_path=FIGURES_DIR / "pr_baseline_test.png")
    error_analysis(splits.test, y_test, test_pred).to_csv(
        REPORTS_DIR / "errors_baseline.csv", index=False)

# ── Transformer
with Timer("evaluate transformer", log):
    val_pred_t, _ = predict_transformer(splits.val)
    test_pred_t, _ = predict_transformer(splits.test)
    tx = {
        "model_dir": str(TRANSFORMER_DIR),
        "val": {**evaluate("transformer_val", y_val, val_pred_t),
                "by_aspect": evaluate_by_aspect(splits.val, y_val, val_pred_t)},
        "test": {**evaluate("transformer_test", y_test, test_pred_t),
                 "by_aspect": evaluate_by_aspect(splits.test, y_test, test_pred_t)},
    }
    results["transformer"] = tx
    log.info(f"transformer test: acc={tx['test']['accuracy']:.4f} macro_f1={tx['test']['macro_f1']:.4f}")
    plot_confusion_matrix(y_test, test_pred_t, title="transformer_test",
                          out_path=FIGURES_DIR / "cm_transformer_test.png")
    plot_per_class_pr(tx["test"]["report"], title="transformer_test",
                      out_path=FIGURES_DIR / "pr_transformer_test.png")
    error_analysis(splits.test, y_test, test_pred_t).to_csv(
        REPORTS_DIR / "errors_transformer.csv", index=False)

## 9. Business insights — NSS, Word Clouds, LDA Topics

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud

_STOPWORDS = {
    "the","a","an","and","or","but","in","on","at","to","for","of","with","is","it",
    "i","you","we","they","this","that","was","are","be","have","has","had","do",
    "did","not","from","by","as","so","if","my","me","he","she","his","her","our",
    "your","their","its","will","would","can","could","should","there","here",
    "what","which","who","how","when","where","why","all","any","some","no","more",
    "also","just","up","out","about","than","then","been","were","am","into",
    "very","get","got","one","two","like","go","use","used","amp","re","s","t","m",
}


def _tokenize(text: str) -> list[str]:
    return [t for t in re.findall(r"\b[a-z]{3,}\b", str(text).lower()) if t not in _STOPWORDS]


def plot_nss_trend(df: pd.DataFrame, out_path: Path) -> None:
    work = df.copy()
    work["created_at_date"] = pd.to_datetime(work["created_at_date"], errors="coerce")
    work = work.dropna(subset=["created_at_date"])
    if work.empty:
        return
    work["date"] = work["created_at_date"].dt.date.astype(str)
    grp = work.groupby("date").agg(
        rows=("sentiment", "size"),
        pos=("sentiment", lambda s: (s == "Positive").sum()),
        neg=("sentiment", lambda s: (s == "Negative").sum()),
    )
    grp["nss"] = (grp["pos"] - grp["neg"]) / grp["rows"].clip(lower=1) * 100
    plt.figure(figsize=(12, 4))
    sns.lineplot(data=grp.reset_index(), x="date", y="nss", marker="o")
    plt.axhline(0, color="red", linestyle="--", linewidth=1)
    plt.title("Daily Net Sentiment Score (NSS)")
    plt.ylabel("NSS (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    plt.close()


def plot_nss_by_aspect(df: pd.DataFrame, out_path: Path) -> None:
    grp = df.groupby("aspect").agg(
        rows=("sentiment", "size"),
        pos=("sentiment", lambda s: (s == "Positive").sum()),
        neg=("sentiment", lambda s: (s == "Negative").sum()),
    )
    grp["nss"] = (grp["pos"] - grp["neg"]) / grp["rows"].clip(lower=1) * 100
    grp = grp.sort_values("nss")
    plt.figure(figsize=(11, 6))
    colors = ["#F44336" if v < 0 else "#FF9800" if v < 30 else "#4CAF50" for v in grp["nss"]]
    plt.barh(grp.index, grp["nss"], color=colors)
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Net Sentiment Score by Aspect")
    plt.xlabel("NSS (%)")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    plt.close()


def plot_wordcloud(texts: list[str], *, title: str, colormap: str, out_path: Path) -> None:
    tokens = [tok for t in texts for tok in _tokenize(t)]
    if not tokens:
        return
    wc = WordCloud(
        width=900, height=450, background_color="white",
        colormap=colormap, max_words=WORDCLOUD_MAX_WORDS, collocations=False,
    ).generate_from_frequencies(Counter(tokens))
    plt.figure(figsize=(11, 5.5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    plt.close()


def plot_topics_lda(texts: list[str], *, title: str, out_path: Path,
                    n_topics: int = LDA_N_TOPICS, n_top_words: int = LDA_N_TOP_WORDS) -> None:
    if len(texts) < n_topics * 2:
        return
    vec = CountVectorizer(
        max_features=4000, stop_words=list(_STOPWORDS),
        token_pattern=r"\b[a-z]{3,}\b", min_df=5,
    )
    X = vec.fit_transform(texts)
    if X.shape[1] == 0:
        return
    feature_names = vec.get_feature_names_out()
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=RANDOM_SEED,
                                    max_iter=15, learning_method="batch").fit(X)
    fig, axes = plt.subplots(2, (n_topics + 1) // 2, figsize=(14, 7))
    axes = axes.flatten()
    for i, comp in enumerate(lda.components_):
        idx = comp.argsort()[: -n_top_words - 1 : -1]
        axes[i].barh([feature_names[j] for j in idx][::-1], comp[idx][::-1], color="#3F51B5")
        axes[i].set_title(f"Topic {i + 1}")
    for j in range(len(lda.components_), len(axes)):
        axes[j].axis("off")
    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    plt.close()


with Timer("NSS + wordclouds + topics", log):
    plot_nss_trend(df, FIGURES_DIR / "nss_trend.png")
    plot_nss_by_aspect(df, FIGURES_DIR / "nss_by_aspect.png")
    pos_texts = df.loc[df["sentiment"] == "Positive", "review_text"].astype(str).tolist()
    neg_texts = df.loc[df["sentiment"] == "Negative", "review_text"].astype(str).tolist()
    plot_wordcloud(pos_texts, title="Positive Reviews", colormap="Greens",
                   out_path=FIGURES_DIR / "wordcloud_positive.png")
    plot_wordcloud(neg_texts, title="Negative Reviews", colormap="Reds",
                   out_path=FIGURES_DIR / "wordcloud_negative.png")
    plot_topics_lda(pos_texts, title="LDA Topics — Positive",
                    out_path=FIGURES_DIR / "topic_lda_positive.png")
    plot_topics_lda(neg_texts, title="LDA Topics — Negative",
                    out_path=FIGURES_DIR / "topic_lda_negative.png")

## 10. Save results

In [ ]:
save_json(results, REPORTS_DIR / "results.json")
log.info(f"results → {REPORTS_DIR / 'results.json'}")

summary = pd.DataFrame([
    {"model": "baseline", "split": "val",
     "acc": results["baseline"]["val"]["accuracy"],
     "macro_f1": results["baseline"]["val"]["macro_f1"]},
    {"model": "baseline", "split": "test",
     "acc": results["baseline"]["test"]["accuracy"],
     "macro_f1": results["baseline"]["test"]["macro_f1"]},
    {"model": "transformer", "split": "val",
     "acc": results["transformer"]["val"]["accuracy"],
     "macro_f1": results["transformer"]["val"]["macro_f1"]},
    {"model": "transformer", "split": "test",
     "acc": results["transformer"]["test"]["accuracy"],
     "macro_f1": results["transformer"]["test"]["macro_f1"]},
])
summary

## 11. Inference demo — rule-based aspect → ABSA sentiment

In [ ]:
def predict(reviews: list[str], *, use_transformer: bool = True) -> list[dict]:
    aspects = [extract_aspect(t) for t in reviews]
    if use_transformer:
        df_in = pd.DataFrame({"review_text": reviews, "aspect": aspects})
        labels, probs = predict_transformer(df_in)
        confidences = probs.max(axis=1).tolist()
    else:
        pairs = [format_pair(a, t) for a, t in zip(aspects, reviews)]
        proba = baseline.predict_proba(pairs)
        idx = proba.argmax(axis=1)
        labels = [str(baseline.classes_[i]) for i in idx]
        confidences = [float(proba[i, idx[i]]) for i in range(len(idx))]
    return [
        {"text": t[:200], "aspect": a, "sentiment": s, "confidence": round(float(c), 4)}
        for t, a, s, c in zip(reviews, aspects, labels, confidences)
    ]


demo = [
    "The BTS was packed during rush hour, no place to stand",
    "Tickets are too expensive for daily commute",
    "Stations are clean and the staff are very helpful",
    "Trains are always on time, very reliable",
]
predict(demo, use_transformer=True)

## 12. Final predictions CSV — aspect + sentiment for every row

In [ ]:
# Predict on the full dataset and save a final CSV with aspect + sentiment
with Timer("predict on full dataset", log):
    full = df.reset_index(drop=True).copy()
    full["aspect_pred"] = [extract_aspect(t) for t in full["review_text"].astype(str)]

    # Use rule-based aspect for inference (consistent with deployment)
    df_in = full[["review_text"]].copy()
    df_in["aspect"] = full["aspect_pred"]
    labels, probs = predict_transformer(df_in)

    full["sentiment_pred"] = labels
    full["sentiment_confidence"] = probs.max(axis=1).round(4)

# Keep only the meaningful columns; drop helper text_id
out_cols = [
    "review_text", "review_rating", "source", "created_at_date",
    "bts_line", "reviewer_hometown",
    "aspect", "sentiment",                        # original (training) labels
    "aspect_pred", "sentiment_pred", "sentiment_confidence",  # model predictions
]
out_cols = [c for c in out_cols if c in full.columns]

final_csv = REPORTS_DIR / "all_reviews_predicted.csv"
full[out_cols].to_csv(final_csv, index=False)
log.info(f"saved → {final_csv}  ({len(full):,} rows)")

full[out_cols].head(5)